# ECERC merged: Limitation 1 (WavLM + VAD) + Limitation 3 (CACG + temperature scaling), IEMOCAP
Settings: Accelerator **GPU T4 x2**. Internet is not needed (WavLM features are precomputed).

Inputs:
- `iemocap-ecerc-data`: the IEMOCAP pickles, `iemocap_aux_labels.pkl` **and** `iemocap_wavlm.pkl`
- `ecerc-merged-code`: the contents of the `Merged_L1_L3` folder

In [ ]:
import glob, os, shutil, subprocess
def find(name):
    hits = glob.glob('/kaggle/input/**/' + name, recursive=True)
    assert hits, name + ' not found in /kaggle/input: attach the dataset that contains it'
    return hits[0]
DATA = os.path.dirname(find('IEMOCAP_features.pkl'))
find('iemocap_aux_labels.pkl')
WAVLM = find('iemocap_wavlm.pkl')
CODE_SRC = os.path.dirname(find('train_merged.py'))
CODE, OUT = '/kaggle/working/code', '/kaggle/working/merged_run'
shutil.copytree(CODE_SRC, CODE, dirs_exist_ok=True)   # /kaggle/input is read-only
print('DATA  =', DATA); print('WAVLM =', WAVLM); print('CODE  =', CODE)
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)

### 1. Sanity checks (about 1 min)
Check 1 is skipped here (it needs the authors' model file); checks 2 and 3 must print values and end with `OK`.

In [ ]:
!cd {CODE} && python test_merge.py --data_dir {DATA} --audio_feat {WAVLM} 2>&1 | grep -v -i warn

### 2. Timing run: one seed of the merged model
Should take about 1 minute. The last line shows test wF1, the pair errors and ECE before and after temperature scaling.

In [ ]:
!cd {CODE} && python train_merged.py --data_dir {DATA} --output_dir {OUT} --tag CACG_W_VAD --seed 1 \
    --gate cacg --gate_loss none --lambda_vad 1.0 --audio_feat {WAVLM} --save_ckpt | tail -3

### 3. Main sweep: 4 configs x 20 seeds (about 45 min)
- `CTRL`: original ECERC
- `W_VAD`: Limitation 1 alone
- `CACG`: Limitation 3 alone
- `CACG_W_VAD`: merged model (checkpoints saved)

In [ ]:
!cd {CODE} && python run_sweep.py --data_dir {DATA} --output_dir {OUT} --gpus 0,1 --seeds 1-20 \
    --configs CACG_W_VAD --audio_feat {WAVLM} --extra=--save_ckpt
!cd {CODE} && python run_sweep.py --data_dir {DATA} --output_dir {OUT} --gpus 0,1 --seeds 1-20 \
    --configs CTRL,W_VAD,CACG --audio_feat {WAVLM}

### 4. Summaries
The merged model against the baseline, against each limitation alone.

In [ ]:
!cd {CODE} && python summarize_merged.py {OUT} CTRL --data_dir {DATA} | tee {OUT}/summary_vs_CTRL.txt
!cd {CODE} && python summarize_merged.py {OUT} W_VAD --data_dir {DATA} | tee {OUT}/summary_vs_W_VAD.txt
!cd {CODE} && python summarize_merged.py {OUT} CACG --data_dir {DATA} | tee {OUT}/summary_vs_CACG.txt

### 5. Final checkpoint: the CACG_W_VAD seed with the best **validation** wF1

In [ ]:
import json
rs = [json.load(open(f)) for f in glob.glob(OUT + '/result_CACG_W_VAD_s*.json')]
best = max(rs, key=lambda r: r['valid_wf1'])
os.makedirs(OUT + '/final', exist_ok=True)
for f in ['ECERC_{}.pkl', 'VADhead_{}.pkl', 'result_{}.json']:
    shutil.copy(os.path.join(OUT, f.format(best['run'])), OUT + '/final/')
print('final checkpoint:', best['run'], '| valid wF1 %.2f | test wF1 %.2f | ECE %.2f -> %.2f (T=%.2f)' % (
    best['valid_wf1'], best['test_wf1'], best['test_ece'], best['test_ece_ts'], best['ts_temperature']))

### 6. Package results (download from the Output tab)
Per-seed checkpoints are left out of the zip; the chosen one is in `final/`.

In [ ]:
!cd /kaggle/working && zip -qr merged_run.zip merged_run -x 'merged_run/ECERC_*' 'merged_run/VADhead_*' && ls -lh merged_run.zip